# Bark: Multilingual & Expressive TTS

## What is Bark?
Bark (by Suno AI) is a **transformer-based TTS** that can:
- Speak in **13+ languages** (including Vietnamese!)
- Add **emotions**: laughing, sighing, hesitation
- Generate **music** and **sound effects** in speech

### Architecture (3-stage pipeline)
```
Text ──▶ [Semantic Model]  ──▶ Semantic Tokens
              (GPT-like)           │
                                   ▼
         [Coarse Acoustic] ──▶ Coarse Tokens
              (GPT-like)           │
                                   ▼
         [Fine Acoustic]   ──▶ Fine Tokens ──▶ [EnCodec] ──▶ 🔊 Audio
              (GPT-like)                        (decoder)
```

**Stage 1**: Text → semantic tokens (captures *what* is said)  
**Stage 2**: Semantic → coarse acoustic (captures *how* it sounds)  
**Stage 3**: Coarse → fine acoustic (adds detail/quality)  

Each stage is a separate GPT-like transformer. This is what makes Bark so expressive!

---
**Kaggle Setup**: GPU T4 x2

## Step 1: Install & Load

In [ ]:
!pip install -q transformers accelerate scipy protobuf>=5.26.1

In [ ]:
import torch
import numpy as np
import time
from transformers import AutoProcessor, BarkModel
from IPython.display import Audio, display

device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# Using bark-small for speed. Use "suno/bark" for full quality
MODEL = "suno/bark-small"

print(f"Loading {MODEL}...")
t0 = time.time()
processor = AutoProcessor.from_pretrained(MODEL)
model = BarkModel.from_pretrained(MODEL, torch_dtype=torch.float16).to(device)
print(f"Loaded in {time.time()-t0:.1f}s")

SAMPLE_RATE = model.generation_config.sample_rate
print(f"Sample rate: {SAMPLE_RATE} Hz")

## Step 2: Basic Speech Generation

Bark uses **voice presets** — pre-recorded speaker embeddings.  
Format: `v2/{language}_speaker_{0-9}`

Available languages:
| Code | Language | Code | Language |
|------|----------|------|----------|
| en | English | zh | Chinese |
| de | German | ja | Japanese |
| fr | French | ko | Korean |
| es | Spanish | hi | Hindi |
| it | Italian | pt | Portuguese |
| pl | Polish | ru | Russian |
| tr | Turkish | | |

In [ ]:
def speak(text, voice_preset="v2/en_speaker_6", label=None):
    """Generate speech with Bark."""
    inputs = processor(text, voice_preset=voice_preset, return_tensors="pt").to(device)
    
    t0 = time.time()
    with torch.no_grad():
        audio = model.generate(**inputs)
    gen_time = time.time() - t0
    
    audio_np = audio.cpu().numpy().squeeze()
    duration = len(audio_np) / SAMPLE_RATE
    
    if label:
        print(f"\n{'='*50}")
        print(f"🎙️ {label}")
    print(f"Voice: {voice_preset} | Duration: {duration:.1f}s | Gen time: {gen_time:.1f}s")
    display(Audio(audio_np, rate=SAMPLE_RATE))
    return audio_np

# Try it!
_ = speak(
    "Hello! I'm Bark, a text to speech model that can speak in many languages.",
    voice_preset="v2/en_speaker_6",
    label="English Speaker 6"
)

## Step 3: Expressive Speech — Emotions & Sound Effects

Bark supports special tokens in text:

| Token | Effect |
|-------|--------|
| `[laughs]` | Laughter |
| `[sighs]` | Sighing |
| `[gasps]` | Gasping |
| `[clears throat]` | Throat clearing |
| `...` | Hesitation/pause |
| `—` | Longer pause |
| `♪` | Singing |
| ALL CAPS | Emphasis |

In [ ]:
# Expressive examples
expressive_examples = [
    ("Laughing",   "v2/en_speaker_9", 
     "Oh my god [laughs] this is SO funny! I can't believe it actually works [laughs]"),
    
    ("Hesitant",   "v2/en_speaker_6",
     "Well... I'm not really sure about this... but I think... maybe we should try it?"),
    
    ("Dramatic",   "v2/en_speaker_3",
     "[sighs] It was a dark and stormy night — the wind was HOWLING through the trees."),
    
    ("Singing",    "v2/en_speaker_9",
     "♪ Twinkle twinkle little star, how I wonder what you are ♪"),
]

for label, preset, text in expressive_examples:
    _ = speak(text, voice_preset=preset, label=label)

## Step 4: Multilingual — Same Model, Many Languages

In [ ]:
multilingual = [
    ("Vietnamese 🇻🇳",  "v2/vi_speaker_0",
     "Xin chào! Tôi là trợ lý giọng nói. Tôi có thể nói tiếng Việt rất tự nhiên."),
    
    ("Chinese 🇨🇳",     "v2/zh_speaker_5",
     "你好！欢迎使用语音合成系统。这是一个非常有趣的技术。"),
    
    ("Japanese 🇯🇵",    "v2/ja_speaker_3",
     "こんにちは！音声合成の世界へようこそ。とても自然な声で話すことができます。"),
    
    ("French 🇫🇷",      "v2/fr_speaker_5",
     "Bonjour! Je suis un modèle de synthèse vocale. Je peux parler français."),
    
    ("German 🇩🇪",      "v2/de_speaker_3",
     "Hallo! Ich bin ein Sprachsynthese-Modell. Ich kann Deutsch sprechen."),
]

for label, preset, text in multilingual:
    _ = speak(text, voice_preset=preset, label=label)

## Step 5: Compare All English Speakers

In [ ]:
test_text = "The future of artificial intelligence is incredibly exciting."

print("Comparing all English voice presets:")
for i in range(10):
    preset = f"v2/en_speaker_{i}"
    _ = speak(test_text, voice_preset=preset, label=f"Speaker {i}")

## Step 6: Long Text with Consistent Voice

In [ ]:
import re

def speak_long(text, voice_preset="v2/en_speaker_6", pause_sec=0.5):
    """Generate long speech by splitting into sentences."""
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    sentences = [s for s in sentences if s.strip()]
    
    chunks = []
    silence = np.zeros(int(SAMPLE_RATE * pause_sec))
    
    for i, sent in enumerate(sentences):
        print(f"  [{i+1}/{len(sentences)}] {sent[:60]}...")
        inputs = processor(sent, voice_preset=voice_preset, return_tensors="pt").to(device)
        with torch.no_grad():
            audio = model.generate(**inputs)
        chunks.append(audio.cpu().numpy().squeeze())
        chunks.append(silence)
    
    full = np.concatenate(chunks)
    print(f"\nTotal: {len(full)/SAMPLE_RATE:.1f}s")
    display(Audio(full, rate=SAMPLE_RATE))
    return full

paragraph = (
    "Text to speech technology has evolved dramatically in recent years. "
    "Modern neural models can produce voices that sound remarkably human. "
    "Bark is unique because it captures emotions and non-verbal sounds. "
    "This makes the generated speech feel alive and natural."
)

_ = speak_long(paragraph, voice_preset="v2/en_speaker_6")

## Key Takeaways

**Pros:**
- 13+ languages with one model
- Expressive: laughs, sighs, singing
- No fine-tuning needed

**Cons:**
- Slower than Parler-TTS
- Voice consistency can drift in long texts
- No voice cloning (fixed presets only)
- Can hallucinate/repeat words

**When to use**: Multilingual content, expressive narration, creative projects

**Next**: Try `03_xtts_v2/` for voice cloning, or `04_xtts_finetune/` to train on your own voice.